# NB7 – CafeBERT (seed 42 và 22): theo báo/năm và tập test đã loại trùng
**Runtime:** CPU (mặc định là đủ)  ·  **Thời gian ước tính:** khoảng 5 phút

Notebook tự chứa: code được nhúng sẵn (các ô `%%writefile`), không cần upload file.
Có thể chạy **song song** với các notebook khác. Chọn *Runtime → Run all*.
Kết quả được chép vào `MyDrive/PLOS_revision/NB7_CPU_cafebert_2seeds/`.

Dùng các file đã có trên Drive: dự đoán CafeBERT seed 42/22 (NB3), corpus_index.pkl, duplicates_test_flags.pkl. Không cần GPU.

In [ ]:
# 1) Drive (kết quả được chép vào đây) + 2) đăng nhập Hugging Face (tài khoản BaoNhan)
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
RESULT_DIR = "/content/drive/MyDrive/PLOS_revision/NB7_CPU_cafebert_2seeds"
os.makedirs(RESULT_DIR, exist_ok=True)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")      # Colab secret tên HF_TOKEN (nếu có)
except Exception:
    from huggingface_hub import login
    login()                                                 # nếu chưa có secret: dán token vào ô hiện ra
from huggingface_hub import whoami
try:
    print("Đã đăng nhập Hugging Face:", whoami()["name"])
except Exception:
    raise SystemExit("CHƯA ĐĂNG NHẬP Hugging Face. Bấm biểu tượng chìa khóa (Secrets) bên trái, "
                     "bật 'Notebook access' cho HF_TOKEN, rồi chạy lại ô này.")
!pip -q install scikit-learn

## Code (nhúng sẵn – không cần sửa)

In [ ]:
%%writefile step7_cafebert_2seeds.py
"""STEP 7 - CafeBERT article-level analyses with the two retained seeds (42, 22). CPU, ~5 min.

Inputs (already on Drive from earlier notebooks):
  PLOS_revision/*/step4/G1_cafebert_seed{42,22}_pred.npz      (NB3, G1)
  PLOS_revision/*/revision_data/preds/PhoBERT_seed42.npz      (label order reference)
  PLOS_revision/*/revision_data/corpus_index.pkl              (publisher, year)
  PLOS_revision/*/revision_results/duplicates_test_flags.pkl  (NB2)
Outputs (PLOS_revision/NB7_CPU_cafebert_2seeds/):
  cafebert_per_source_year.csv, cafebert_duplicates_rescored.csv, step7_report.txt
"""
import glob
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

PLOS = Path(os.environ.get("PLOS_ROOT", "/content/drive/MyDrive/PLOS_revision"))
OUT = Path(os.environ.get("STEP7_OUT", PLOS / "NB7_CPU_cafebert_2seeds"))
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 22)
lines = []


def log(m=""):
    print(m, flush=True)
    lines.append(str(m))
    (OUT / "step7_report.txt").write_text("\n".join(lines), encoding="utf-8")


def find_one(pattern, what):
    hits = sorted(glob.glob(pattern))
    if not hits:
        log(f"STOP: {what} not found ({pattern})")
        sys.exit(1)
    log(f"{what}: {hits[0]}")
    return hits[0]


f1 = lambda y, p: precision_recall_fscore_support(y, p, average="macro", zero_division=0)[2]
R = {s: dict(np.load(find_one(f"{PLOS}/*/step4/G1_cafebert_seed{s}_pred.npz", f"CafeBERT seed {s}")))
     for s in SEEDS}
y = np.load(find_one(f"{PLOS}/*/revision_data/preds/PhoBERT_seed42.npz", "PhoBERT seed 42"))["true"].astype(int)
for s in SEEDS:
    assert np.array_equal(R[s]["true"].astype(int), y), f"seed {s}: label order differs"
    log(f"CafeBERT seed {s}: n={len(y)}, correct={int(np.sum(R[s]['pred'] == y))}, "
        f"accuracy={np.mean(R[s]['pred'] == y):.6f}, macro_f1={f1(y, R[s]['pred']):.6f}")

ci = pd.read_pickle(find_one(f"{PLOS}/*/revision_data/corpus_index.pkl", "corpus_index.pkl"))
t = ci[ci.split == "test"].sort_values("pos")
assert np.array_equal(t["label"].to_numpy(), y), "corpus_index order differs from predictions"
rows = []
for col in ("source", "year"):
    vals = t[col].to_numpy()
    for v in sorted(pd.Series(vals).dropna().unique()):
        m = vals == v
        f1s = [f1(y[m], R[s]["pred"][m]) for s in SEEDS]
        accs = [np.mean(y[m] == R[s]["pred"][m]) for s in SEEDS]
        rows.append(dict(by=col, group=v, model="CafeBERT", n=int(m.sum()), macro_f1_mean=np.mean(f1s),
                         macro_f1_sd=np.std(f1s), acc_mean=np.mean(accs), acc_sd=np.std(accs)))
psy = pd.DataFrame(rows)
psy.to_csv(OUT / "cafebert_per_source_year.csv", index=False)
log("publisher / year (CafeBERT, seeds 42 and 22):\n" + psy.to_csv(index=False))

fl = pd.read_pickle(find_one(f"{PLOS}/*/revision_results/duplicates_test_flags.pkl", "duplicates_test_flags.pkl"))
dup = fl["any_duplicate"].to_numpy()
out = []
for name, m in [("full_test", np.ones_like(dup)), ("deduplicated_test", ~dup)]:
    f1s = [f1(y[m], R[s]["pred"][m]) for s in SEEDS]
    accs = [np.mean(y[m] == R[s]["pred"][m]) for s in SEEDS]
    out.append(dict(model="CafeBERT", subset=name, n=int(m.sum()), macro_f1_mean=np.mean(f1s),
                    macro_f1_sd=np.std(f1s), acc_mean=np.mean(accs), acc_sd=np.std(accs)))
out = pd.DataFrame(out)
out.to_csv(OUT / "cafebert_duplicates_rescored.csv", index=False)
log("de-duplicated test (CafeBERT, seeds 42 and 22):\n" + out.to_csv(index=False))
log("Done.")

## Dùng lại kết quả cũ trên Drive

In [ ]:
# Dùng lại kết quả của lần chạy trước trên Drive (nếu có) để bỏ qua các bước đã xong
import shutil, glob, os
for d in ["revision_data", "revision_data/preds", "revision_results"]:
    os.makedirs(d, exist_ok=True)
    for f in glob.glob(f"{RESULT_DIR}/{d}/*"):
        if os.path.isfile(f) and not os.path.exists(f"{d}/{os.path.basename(f)}"):
            shutil.copy(f, f"{d}/")
have = lambda p: os.path.exists(p)
print("preds:", len(glob.glob("revision_data/preds/*.npz")), "| corpus_index:", have("revision_data/corpus_index.pkl"),
      "| step2:", have("revision_results/step2_report.txt"))

## Chạy

In [ ]:
os.environ["STEP7_OUT"] = RESULT_DIR
!python step7_cafebert_2seeds.py

## Lưu kết quả

In [ ]:
# Chép kết quả sang Drive và tạo file zip để tải về
import shutil, glob
for d in ["revision_results", "revision_data"]:
    for f in glob.glob(f"{d}/*.txt") + glob.glob(f"{d}/*.csv") + glob.glob(f"{d}/*.tex") + \
             glob.glob(f"{d}/*.pdf") + glob.glob(f"{d}/*.tif") + glob.glob(f"{d}/*.pkl"):
        if os.path.getsize(f) < 500e6:
            os.makedirs(f"{RESULT_DIR}/{d}", exist_ok=True)
            shutil.copy(f, f"{RESULT_DIR}/{d}/")
os.makedirs(f"{RESULT_DIR}/revision_data/preds", exist_ok=True)
for f in glob.glob("revision_data/preds/*.npz"):
    shutil.copy(f, f"{RESULT_DIR}/revision_data/preds/")
print("Đã chép vào", RESULT_DIR)